# **Plotting the data $w(\theta)$ for a given dataset**

This notebook plots the data angular correlation function $w(\theta)$ for a given dataset or a set of datasets.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
from utils_data import RedshiftDistributions, GetThetaLimits, WThetaDataCovariance

tracer = "LRG"
# tracer = "LRG+ELG"
# tracer = "ELG"
tracer = "ELG2"
tracer = "QSO"


if tracer in ["ELG2", "QSO"]:
    delta_z = 0.05
else:
    delta_z = 0.02

if tracer in ["ELG", "ELG2", "QSO"]:
    theta_width = 3
else:
    theta_width = 6

theta_data = {}
wtheta_data = {}
cov = {}

for recon in ["pre", "post"]:
# for recon in ["post"]:

    dataset = f"DESIY1_{tracer}_EZ_ffa_{recon}_deltaz{delta_z}"
    dataset = f"DESIY1_{tracer}_Abacus_altmtl_{recon}_deltaz{delta_z}"
    # dataset = f"DESIY1_{tracer}_{recon}_deltaz{delta_z}"
    
    nz_flag = "mocks"
    
    nz = RedshiftDistributions(dataset=dataset, nz_flag=nz_flag, code_path=code_path)
    theta_min, theta_max = GetThetaLimits(dataset=dataset, nz_flag=nz_flag, dynamical_theta_limits=True, theta_width=theta_width, code_path=code_path).get_theta_limits()
    
    bins_removed = []
    
    # 1. Arguments
    config = {
        "dataset": dataset,
        "weight_type": 1, # it will not be used...
        "mock_id": "mean",
        "nz_flag": nz_flag,
        "cov_type": "mocks",
        "cosmology_covariance": "desifid",
        "delta_theta": 0.2,
        "theta_min": theta_min,
        "theta_max": theta_max,
        "bins_removed": bins_removed,
        "diag_only": "n",
        "remove_crosscov": "n",
        "code_path": code_path,
    }
    
    wtheta_data_covariance = WThetaDataCovariance(**config)
    
    theta_data[dataset], wtheta_data[dataset], cov[dataset] = wtheta_data_covariance.process()


In [ ]:
# Plot the w(theta)
nbins_eff = nz.nbins - len(bins_removed)
fig, axs = plt.subplots(nbins_eff, 1, figsize=(8, 2 * (nbins_eff)), sharex=True)
axs = np.atleast_1d(axs)
i = 0
for bin_z in range(nz.nbins):
    if bin_z not in bins_removed:
        ax = axs[i]

        for dataset in theta_data.keys():
            ax.errorbar(
                theta_data[dataset][bin_z] * 180 / np.pi,
                100 * (theta_data[dataset][bin_z] * 180 / np.pi) ** 2 * wtheta_data[dataset][bin_z],
                yerr=100 * (theta_data[dataset][bin_z] * 180 / np.pi) ** 2 * np.sqrt(np.diag(cov[dataset]))[sum(len(theta_data[dataset][bin_z2]) for bin_z2 in range(bin_z)):sum(len(theta_data[dataset][bin_z2]) for bin_z2 in range(bin_z + 1))],
                capsize=4, capthick=1.5,
                marker="D", markersize=6, markeredgewidth=1.2, linestyle="none",
                label=fr"\texttt{{{dataset}}}",
            )
        ax.set_ylabel(r"$10^2 \times \theta^2w(\theta)$", fontsize=22)
        ax.tick_params(axis="x", labelsize=18)
        ax.tick_params(axis="y", labelsize=18)
        z_edge = nz.z_edges[bin_z]
        if "DESIY1" in dataset:
            ax.text(0.13, 0.1, f"{z_edge[0]:.2f} $< z <$ {z_edge[1]:.2f}", ha="center", va="center", transform=ax.transAxes, fontsize=18)
        else:
            ax.text(0.13, 0.1, f"{z_edge[0]} $< z <$ {z_edge[1]}", ha="center", va="center", transform=ax.transAxes, fontsize=18)
            
        if i == 0:
            ax.legend(loc="upper left", fontsize=18)
        if i == nbins_eff - 1:
            ax.set_xlabel(r"$\theta$ (deg)", fontsize=22)
        i += 1
if nbins_eff != 1:
    fig.tight_layout()
    